In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings
import os
import sys

warnings.filterwarnings("ignore")

os.environ["R_HOME"] = "/home/bio/miniconda3/envs/spCLUE/lib/R"
# os.environ["R_HOME"] = "/home/lxx/.conda/envs/r4Base/lib/R"
# os.environ["R_USER"] = "/home/lxx/.local/lib/python3.9/site-packages/rpy2"
spCLUE_ROOT_PATH = "/home/bio/lhz/spatialGDC"
sys.path.append(spCLUE_ROOT_PATH)
import spCLUE

spCLUE.fix_seed(0)

/home/bio/miniconda3/envs/spCLUE/lib/python3.9/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [ ]:
sample_name="151671"
# 1. 预处理数据
adata = spCLUE.preprocess_data(input_dir="../dataset/DLPFC/151671/") 

# 🔥 关键修正：先剔除没有标签的细胞，再建图！
adata = adata[adata.obs.Region.notna()].copy() # 使用 .copy() 避免 View 警告

n_clusters = 5 if sample_name in [str(151669 + x) for x in range(4)] else 7

# 2. 现在基于过滤后的 4093 个点生成 PCA 和 图
adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
g_spatia = spCLUE.prepare_graph(adata, "spatial")
g_expr = spCLUE.prepare_graph(adata, "expr")
graph_dict = {"spatial": g_spatia, "expr": g_expr}

In [2]:
import itertools
import copy

# 1. 定义搜索空间 (根据之前的分析，权重建议往小了试)
search_space = {
    'alpha': [0.01, 0.1, 0.5],      # 簇级对比权重
    'beta': [0.005, 0.01, 0.05],    # 实例级对比权重 (之前0.1太高了，往调小)
    'warmup': [50, 100]             # 预热轮数
}

# 生成所有组合
keys, values = zip(*search_space.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

results = []
best_ari = 0
best_params = None

print(f"开始网格搜索，共 {len(param_combinations)} 组配置...")

for i, params in enumerate(param_combinations):
    print(f"\n[实验 {i+1}/{len(param_combinations)}] 参数: {params}")
    
    # 重新初始化模型，保证实验独立
    test_model = spCLUE.spCLUE(adata.obsm["X_pca"], graph_dict, n_clusters)
    
    # 训练模型
    _, adata.obsm["spCLUE"] = test_model.train(
        warmup_epochs=params['warmup'], 
        alpha_dcd=params['alpha'], 
        beta_rngpa=params['beta']
    )
    
    # 执行聚类与细化
    spCLUE.clustering(
        adata,
        n_clusters,
        key="spCLUE",
        refinement=True,
        cluster_methods="mclust"
    )
    
    # 计算当前 ARI
    # 确保只计算有标签的区域
    valid_adata = adata[adata.obs.Region.notna()]
    current_ari = adjusted_rand_score(valid_adata.obs["Region"], valid_adata.obs["mclust_refined"])
    
    print(f">>> 实验 {i+1} 结束, ARI: {current_ari:.4f}")
    
    # 记录结果
    results.append({**params, 'ari': current_ari})
    
    # 更新最优配置
    if current_ari > best_ari:
        best_ari = current_ari
        best_params = params
        # 可选：保存当前最好的特征矩阵
        adata.obsm["best_spCLUE_features"] = adata.obsm["spCLUE"].copy()

print("\n" + "="*30)
print(f"网格搜索完成！")
print(f"最高 ARI: {best_ari:.4f}")
print(f"最佳参数组合: {best_params}")
print("="*30)

开始网格搜索，共 18 组配置...

[实验 1/18] 参数: {'alpha': 0.01, 'beta': 0.005, 'warmup': 50}


NameError: name 'adata' is not defined